# Solution 2.2.2 — Data Types and Subsetting

### Path Setup

In [1]:
import os
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw'
FILE_NAME = 'datania_households_raw.csv'
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

# Identifiers and codes must stay as text (preserve leading zeros)
df = pd.read_csv(raw_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

Loaded: (28, 11)


,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
0,HH0001,01,Eastern Province,Kurtosis Bay,Urban,4,45 000,2025-01-10,780,3,42
1,HH0002,02,Northern Province,Vector Hills,Rural,6,Ar 32000,2025-01-11,120,2,39
2,HH0003,03,Central Province,Polaris District,Urban,3,54000,2025-01-12,640,4,33
3,HH0004,04,Southern Province,Lagoon Point,Rural,5,NaN,2025-01-13,80,1,51
4,HH0005,05,Western Province,Gamma Plains,Urban,2,unknown,2025-01-13,520,2,28


---

## Task 1 — DataFrame vs Series

In [2]:
print(type(df))
print(type(df['income_dkw']))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>


In [3]:
df.dtypes

hh_id             object
region_code       object
province_name     object
district          object
urban_rural       object
hh_size            int64
income_dkw        object
survey_date       object
pop_density        int64
education_code     int64
age                int64
dtype: object

**Answers:**

- `income_dkw` is text (`object`/`str`), **not** numeric — pandas falls back to text because of values like `'Ar 32000'`, `'45 000'`, and `'unknown'`.
- `survey_date` is text too. You must convert it with `pd.to_datetime()` before the `.dt` accessor (and the month) become available.
- We forced `region_code` to text with `dtype={'region_code': str}`. Without it, pandas would infer `int64` and silently drop leading zeros (`'01'` → `1`).

---

## Task 2 — Standardise text and fix categories

In [4]:
print(df['urban_rural'].value_counts(dropna=False))
print()
print(df['region_code'].value_counts(dropna=False))

urban_rural
Urban    14
Rural    13
Urbn      1
Name: count, dtype: int64

region_code
05     5
06     5
02     4
03     4
04     4
01     3
1      1
NaN    1
99     1
Name: count, dtype: int64


In [5]:
df['urban_rural'] = df['urban_rural'].replace('Urbn', 'Urban')
df['region_code'] = df['region_code'].replace({'99': np.nan, '1': '01'})

print(df['urban_rural'].value_counts(dropna=False))
print()
print(df['region_code'].value_counts(dropna=False))

urban_rural
Urban    15
Rural    13
Name: count, dtype: int64

region_code
05     5
06     5
01     4
02     4
03     4
04     4
NaN    2
Name: count, dtype: int64


**Answers:**

- `.str.replace('1', '01')` replaces the *character* `'1'` inside every string, so `'01'` → `'001'`, `'16'` → `'016'`, `'21'` → `'021'`. `.replace()` with a dict matches whole values, so only the cell equal to `'1'` becomes `'01'`.
- `region_code` ends with two `NaN`: the originally blank cell (`HH0023`) and the `'99'` coded-missing value (`HH0026`).

---

## Task 3 — Convert text to numbers

In [6]:
df['income_dkw'].unique()

array(['45 000', 'Ar 32000', '54000', nan, 'unknown', '-5000', '120000',
       '38000', '1,200,000', ' 41000 ', '29000', '65000', '87000',
       'Ar 74 000', '51000', '26000', '94000', '33000', '56000', '999999',
       '35000', 'Ar 99000', '61000', '58000', '39000'], dtype=object)

In [7]:
df['income_dkw'] = (
    df['income_dkw']
    .astype('string')
    .str.replace(' ', '', regex=False)
    .str.replace('Ar', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace({'unknown': np.nan, 'NA': np.nan})
)
# Every label is now NaN, so the strict default conversion succeeds
df['income_dkw'] = pd.to_numeric(df['income_dkw'], errors='raise')

df['income_dkw'].describe()

count          24.00
mean      141,541.62
std       297,891.30
min        -5,000.00
25%        37,250.00
50%        55,000.00
75%        77,250.00
max     1,200,000.00
Name: income_dkw, dtype: Float64

**Answers:**

- Four values are `NaN`: the two blanks (`HH0004`, `HH0016`), `'unknown'` (`HH0005`), and `'NA'` (`HH0021`, which pandas already treats as missing at load time).
- Replacing `unknown`/`NA` with `NaN` first leaves only numeric strings and blanks, so `pd.to_numeric()` with the strict default `errors='raise'` converts cleanly. Using `'raise'` is deliberate: if a non-numeric label slipped through it errors immediately, instead of silently hiding the problem.
- They are valid numbers syntactically, so they survive here. They are impossible/sentinel incomes and get recoded to `NaN` as coded-missing values in 2.2.3.

---

## Task 4 — Convert text to dates

In [8]:
df['survey_date'].unique()

array(['2025-01-10', '2025-01-11', '2025-01-12', '2025-01-13',
       '2025-01-14', '03/15/2025', '2025-01-16', '2025-01-17',
       '2025/01/18', '2025-01-19', '2025-13-01', '2025-01-21', nan,
       '2025-01-23', 'not recorded', '2025-01-25', '2025-01-26',
       '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',
       '2025-01-31', '2025-02-01', '2025-02-02', '2025-02-05',
       '2025-02-07'], dtype=object)

In [9]:
date_fixes = {
    'not recorded': np.nan,
    '03/15/2025':   '2025-03-15',   # MM/DD/YYYY
    '2025/01/18':   '2025-01-18',   # slash separators
    '2025-13-01':   '2025-01-13',   # day/month inverted
}
df['survey_date'] = df['survey_date'].replace(date_fixes)

# Every malformed value is fixed or NaN, so the strict default conversion succeeds
df['survey_date'] = pd.to_datetime(df['survey_date'], errors='raise')

df['survey_date'].dtype

dtype('<M8[ns]')

In [10]:
df['survey_date'].dt.month

0    1.00
1    1.00
2    1.00
3    1.00
4    1.00
5    1.00
6    3.00
7    1.00
8    1.00
9    1.00
10   1.00
11   1.00
12   1.00
13   1.00
14    NaN
15   1.00
16    NaN
17   1.00
18   1.00
19   1.00
20   1.00
21   1.00
22   1.00
23   1.00
24   2.00
25   2.00
26   2.00
27   2.00
Name: survey_date, dtype: float64

**Answers:**

- Four values needed attention: `'not recorded'` (not a date → `NaN`), `'03/15/2025'` (MM/DD/YYYY), `'2025/01/18'` (slash separators), and `'2025-13-01'` (month 13 is impossible — day and month inverted). The blank date (`HH0014`) is already missing and parses to `NaT`.
- `NaT` (Not a Time) is the datetime equivalent of `NaN`. It marks a missing/unparseable date and is skipped in datetime aggregations, but it lives only in datetime columns.
- Other handy `.dt` properties: `.dt.year`, `.dt.day`, `.dt.day_name()`, `.dt.quarter`, `.dt.is_month_end`.

---

## Task 5 — Subsetting to inspect problems

In [11]:
df[df['income_dkw'] < 0][['hh_id', 'income_dkw']]

,hh_id,income_dkw
5,HH0006,-5000


In [12]:
df[(df['urban_rural'] == 'Rural') & (df['income_dkw'] > 50000)][['hh_id', 'urban_rural', 'income_dkw']]

,hh_id,urban_rural,income_dkw
13,HH0013,Rural,87000
15,HH0015,Rural,51000
25,HH0025,Rural,61000


In [13]:
target_districts = ['Polaris District', 'North Delta']
df[df['district'].isin(target_districts)][['hh_id', 'district']]

,hh_id,district
2,HH0003,Polaris District
6,HH0007,North Delta
17,HH0017,Polaris District


In [14]:
df[df['district'].str.contains('delta', case=False, na=False)][['hh_id', 'district']]

,hh_id,district
6,HH0007,North Delta
7,HH0008,South Delta


**Answers:**

- One household has negative income: `HH0006` (`-5000`).
- Each condition must be parenthesised because `&`/`|` bind tighter than the comparison operators; without parentheses pandas evaluates `50000 & (df['urban_rural']...)` first and raises an error.
- `na=False` tells `str.contains()` to treat missing values as `False` instead of returning `NaN`, which would break the boolean mask used for filtering.

---

## Task 6 — Save a typed checkpoint to `10_cleaned/`

In [15]:
COLS_OUT = [
    'hh_id', 'region_code', 'province_name', 'district', 'urban_rural',
    'hh_size', 'income_dkw', 'survey_date', 'pop_density', 'education_code', 'age',
]
df_typed = df[COLS_OUT].copy()
print('Typed checkpoint:', df_typed.shape)
df_typed.head()

Typed checkpoint: (28, 11)


,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
0,HH0001,01,Eastern Province,Kurtosis Bay,Urban,4,45000,2025-01-10,780,3,42
1,HH0002,02,Northern Province,Vector Hills,Rural,6,32000,2025-01-11,120,2,39
2,HH0003,03,Central Province,Polaris District,Urban,3,54000,2025-01-12,640,4,33
3,HH0004,04,Southern Province,Lagoon Point,Rural,5,<NA>,2025-01-13,80,1,51
4,HH0005,05,Western Province,Gamma Plains,Urban,2,<NA>,2025-01-13,520,2,28


In [16]:
DATA_CLEAN_DIR = '../../data/10_cleaned'
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

df_typed.to_csv(out_path, index=False)
print('Saved:', out_path)

Saved: ../../data/10_cleaned/datania_households_clean.csv


In [17]:
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded:', check.shape)
print(check.dtypes)
check.head()

Reloaded: (28, 11)
hh_id              object
region_code        object
province_name      object
district           object
urban_rural        object
hh_size             int64
income_dkw        float64
survey_date        object
pop_density         int64
education_code      int64
age                 int64
dtype: object


,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
0,HH0001,01,Eastern Province,Kurtosis Bay,Urban,4,"45,000.00",2025-01-10,780,3,42
1,HH0002,02,Northern Province,Vector Hills,Rural,6,"32,000.00",2025-01-11,120,2,39
2,HH0003,03,Central Province,Polaris District,Urban,3,"54,000.00",2025-01-12,640,4,33
3,HH0004,04,Southern Province,Lagoon Point,Rural,5,NaN,2025-01-13,80,1,51
4,HH0005,05,Western Province,Gamma Plains,Urban,2,NaN,2025-01-13,520,2,28


**Answers:**

- After reloading, `survey_date` comes back as `object` (text). CSV stores everything as plain text and cannot preserve a datetime type — anyone reading the file must re-parse the date.
- `index=False` stops pandas from writing the row index as an extra unnamed column.
- This file is only *typed*: it still contains duplicates, coded-missing sentinels (`999`, `9999`, `99`, `999999`), a sign-entry error (`-5000`), and impossible values (`hh_size` of `0`, `-1`, `99`). Exercise 2.2.3 handles those.